# Exposure Time Calculator

Estimate exposure times for MuSCAT observations with
`muscat_db.exposure`, the same calculator the observation-planning web
page uses. Given a target magnitude, a band, and the observing airmass, it
returns the exposure time that reaches a chosen peak signal level without
saturating the detector.

**Environment:** run with the `prose` conda kernel (`conda activate
prose`). The setup cell makes the `muscat_db` package importable from
source and reads calibration coefficients from `muscat.db`.

In [ ]:
import os, sys, types
from pathlib import Path

# Make the muscat_db package importable from source. The package __init__
# imports python-dotenv, which the prose env does not ship; stub it only if
# it is genuinely absent so the import still succeeds here.
try:
    import dotenv  # noqa: F401
except ModuleNotFoundError:
    _stub = types.ModuleType("dotenv")
    _stub.find_dotenv = lambda *a, **k: ""
    _stub.load_dotenv = lambda *a, **k: False
    sys.modules["dotenv"] = _stub

_here = Path.cwd()
_root = next((p for p in (_here, *_here.parents) if (p / "src" / "muscat_db").is_dir()), None)
assert _root is not None, "run this from inside the muscat-db repo"
_db_path = Path(os.environ.get("MUSCAT_DB_PATH") or _root / "muscat.db").expanduser().resolve()
if not _db_path.is_file():
    raise FileNotFoundError(f"muscat.db not found: {_db_path}")
os.environ["MUSCAT_DB_PATH"] = str(_db_path)
sys.path.insert(0, str(_root / "src"))

import numpy as np
import matplotlib.pyplot as plt
from muscat_db.exposure import calc_exptime, calc_all_bands, calibration_status

print("muscat_db loaded from", _root / "src" / "muscat_db")
print("muscat.db:", _db_path)

## 1. Exposure time for one target

`calc_exptime` returns the exposure needed to reach `target_adu` peak
counts, plus useful diagnostics: the expected stellar FWHM at that focus,
the fraction of the detector full well used, and a saturation flag. A
common safety target is ~50% of full well.

In [ ]:
INSTRUMENT = "muscat3"
FOCUS_MM = 0.0
TARGET_ADU = 40000        # peak counts to aim for
AIRMASS = 1.3

# Approximate Pan-STARRS magnitudes of a medium-brightness host star.
mags = {"gp": 12.5, "rp": 12.0, "ip": 11.8, "zs": 11.5}

print(f"{INSTRUMENT}  focus {FOCUS_MM} mm  airmass {AIRMASS}  target {TARGET_ADU} ADU\n")
header = "{:4s} {:>5s} {:>8s} {:>9s} {:>7s}  saturated".format(
    "band", "mag", "exp (s)", "FWHM(as)", "% well")
print(header)
for band, mag in mags.items():
    r = calc_exptime(
        instrument=INSTRUMENT, band=band, mag=mag,
        focus_mm=FOCUS_MM, target_adu=TARGET_ADU, airmass=AIRMASS,
    )
    print("{:4s} {:5.1f} {:8.1f} {:9.2f} {:7.1f}  {}".format(
        band, mag, r["exptime"], r["fwhm_arcsec"], r["pct_full_well"], r["is_saturated"]))

## 2. Airmass and atmospheric extinction

As a target sinks toward the horizon its light passes through more
atmosphere, so it appears fainter and needs a longer exposure. Extinction
is stronger at shorter wavelengths, so the blue (g) band grows fastest with
airmass while the red (z) band is barely affected.

In [ ]:
airmasses = np.linspace(1.0, 2.5, 30)

fig, ax = plt.subplots(figsize=(9, 5))
for band in ["gp", "rp", "ip", "zs"]:
    exps = [
        calc_exptime(instrument=INSTRUMENT, band=band, mag=mags[band],
                     focus_mm=FOCUS_MM, target_adu=TARGET_ADU, airmass=a)["exptime"]
        for a in airmasses
    ]
    ax.plot(airmasses, exps, label=band)
_ = ax.set_xlabel("Airmass")
_ = ax.set_ylabel("Exposure time (s)")
_ = ax.set_title(f"Exposure vs. airmass ({INSTRUMENT}, focus {FOCUS_MM} mm)")
ax.legend()
fig.tight_layout()
plt.show()

## 3. Focus dependence

MuSCAT is often deliberately defocused to spread starlight over more
pixels: this avoids saturation on bright targets and averages over
pixel-to-pixel sensitivity, at the cost of a larger PSF. A larger focus
offset spreads the light, so a longer exposure is needed for the same peak
ADU.

In [ ]:
focus_values = np.linspace(0.0, 6.0, 25)
band = "rp"
exps = [
    calc_exptime(instrument=INSTRUMENT, band=band, mag=mags[band],
                 focus_mm=f, target_adu=TARGET_ADU, airmass=1.1)["exptime"]
    for f in focus_values
]
fwhm = [
    calc_exptime(instrument=INSTRUMENT, band=band, mag=mags[band],
                 focus_mm=f, target_adu=TARGET_ADU, airmass=1.1)["fwhm_arcsec"]
    for f in focus_values
]

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(focus_values, exps, color="C0")
ax1.set_xlabel("Focus offset (mm)")
ax1.set_ylabel("Exposure time (s)", color="C0")
ax2 = ax1.twinx()
ax2.plot(focus_values, fwhm, color="C1", ls="--")
ax2.set_ylabel("FWHM (arcsec)", color="C1")
_ = ax1.set_title(f"Defocus trade-off ({INSTRUMENT}, {band})")
fig.tight_layout()
plt.show()

## 4. Saturation and comparison stars

The exposure is not set by the target alone. Every star in the field is
exposed for the same time, so a comparison star much brighter than the
target will saturate first and cap the usable exposure, starving the target
of signal. `calc_all_bands` accepts extra sources and reports the
`recommended_exptime` that keeps *every* source below the saturation limit.

In [ ]:
bright_comp = {"label": "bright comparison",
                "mags": {"gp": 10.0, "rp": 9.7, "ip": 9.5, "zs": 9.3}}

alone = calc_all_bands(INSTRUMENT, mags, focus_mm=FOCUS_MM, airmass=AIRMASS, sat_frac=0.5)
with_comp = calc_all_bands(INSTRUMENT, mags, focus_mm=FOCUS_MM, airmass=AIRMASS,
                           sat_frac=0.5, extra_sources=[bright_comp])

print(f"target alone       : recommended exposure {alone['recommended_exptime']:.1f} s")
print(f"with bright comp   : recommended exposure {with_comp['recommended_exptime']:.1f} s")
print("\nthe bright comparison saturates first and forces a shorter exposure,")
print("which is why comparison-star brightness matters when planning (see the")
print("FOV optimization notebook for choosing a good comparison set).")

## 5. Calibration status

The calculator prefers coefficients measured from real frames in
`muscat.db`, falling back to empirical values when a band has not been
calibrated. `calibration_status` reports what is available per instrument.

In [ ]:
for inst in ["muscat3", "muscat4", "sinistro"]:
    st = calibration_status(inst)
    print(f"{inst}: {st['n_bands']} band(s) calibrated")
    for b in st.get("bands", []):
        print(f"    {b['band']:4s}  {b['n_frames']:>5d} frames  {b['n_focus']} focus points  (updated {b['updated_at']})")

## Summary

- `calc_exptime` gives the exposure to reach a target peak ADU, with FWHM
  and saturation diagnostics.
- Exposure grows with airmass (extinction, strongest in the blue) and with
  defocus.
- The brightest source in the field, often a comparison star, caps the
  usable exposure; plan the field and exposure together.
- Coefficients come from `muscat.db` calibration where available, so results
  track the real instrument, and match the observation-planning web page.